# `RunnableWithFallbacks: RunnableSerializable[Input, Output]`

`RunnableWithFallbacks` executes a primary `Runnable` and tries fallback Runnables in order when the primary or an earlier fallback raises a handled exception.

It is normally created through `runnable.with_fallbacks(...)`, although it can also be instantiated directly.

## Type Parameters

```python
Input # Input type accepted by the primary Runnable and every fallback
Output # Output type produced by the primary Runnable and every fallback
```

## Fields

```python
runnable: Runnable[Input, Output] # Primary Runnable executed first
fallbacks: Sequence[Runnable[Input, Output]] # Fallback Runnables tried in order
exceptions_to_handle: tuple[type[BaseException], ...] = (Exception,) # Exception types that trigger fallback execution
exception_key: str | None = None # Dictionary key used to pass the previous handled exception to the next fallback
```

## Constructor

The constructor is generated from the serializable model fields.

```python
RunnableWithFallbacks(
    *,
    runnable: Runnable[Input, Output], # Primary Runnable executed first
    fallbacks: Sequence[Runnable[Input, Output]], # Ordered fallback Runnables
    exceptions_to_handle: tuple[type[BaseException], ...] = (Exception,), # Exceptions that allow fallback execution
    exception_key: str | None = None, # Input dictionary key receiving the previous handled exception
    name: str | None = None, # Optional name used for tracing and debugging
) -> None # Initialize the fallback wrapper
```

When `exception_key` is set, the input must be a dictionary accepted by the primary Runnable and every fallback.

## Public Property

### `runnables`

Returns an iterator containing the primary Runnable followed by every fallback in execution order.

```python
runnables: Iterator[Runnable[Input, Output]] # Primary Runnable followed by ordered fallbacks
```

## Overridden Properties and Methods

### `InputType`

Returns the input type of the primary Runnable.

### `OutputType`

Returns the output type of the primary Runnable.

### `get_input_schema`

Returns the input schema of the primary Runnable.

### `get_output_schema`

Returns the output schema of the primary Runnable.

### `config_specs`

Returns the unique configurable-field specifications collected from the primary Runnable and all fallbacks.

### `is_lc_serializable`

Returns `True`, indicating that the class supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain serialization namespace for Runnable objects.

### `invoke`

Synchronously executes the primary Runnable and then tries each fallback until one succeeds.

An exception outside `exceptions_to_handle` is raised immediately.

When every Runnable fails with a handled exception, the first handled exception is raised.

### `ainvoke`

Asynchronously executes the primary Runnable and then tries each fallback until one succeeds.

It follows the same exception rules as `invoke()`.

### `batch`

Processes each input independently through the primary Runnable and fallbacks.

Successful inputs are removed from later fallback attempts, while only failed inputs continue to the next Runnable.

When `return_exceptions=True`, final failures are returned in the result list.

### `abatch`

Asynchronously processes each input independently through the primary Runnable and fallbacks.

It follows the same per-input and exception behaviour as `batch()`.

### `stream`

Tries the primary Runnable and fallbacks until one successfully produces its first output chunk.

After the first chunk is produced, later errors from that selected stream are raised and do not switch to another fallback.

### `astream`

Asynchronously tries the primary Runnable and fallbacks until one successfully produces its first output chunk.

After the first chunk is produced, later errors from that selected stream are raised and do not switch to another fallback.

### `__getattr__`

Delegates missing attributes to the primary Runnable.

When the delegated attribute is a method returning a new Runnable, the method is applied to the primary Runnable and every fallback, and a new `RunnableWithFallbacks` is returned.

## Exception Behaviour

- Only exceptions listed in `exceptions_to_handle` trigger fallback execution.
- Unhandled exception types are raised immediately.
- Fallbacks are tried from first to last.
- Execution stops as soon as one Runnable succeeds.
- If all attempts fail with handled exceptions, the first handled exception is raised.
- With `exception_key`, the most recent handled exception is inserted into the input dictionary before the next fallback runs.

## Batch Behaviour

- Each batch input may succeed on a different Runnable.
- Successful results retain their original input order.
- Only failed inputs are retried by the next fallback.
- Empty input returns an empty list.
- With `return_exceptions=False`, unhandled exceptions are raised.
- With `return_exceptions=True`, final exceptions are returned alongside successful outputs.

## Streaming Behaviour

Fallback selection occurs only while attempting to obtain the first stream chunk.

Once a Runnable produces its first chunk, it becomes the selected stream for the rest of that execution.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.runnables.fallbacks import RunnableWithFallbacks # Import RunnableWithFallbacks


def primary_operation(number: int) -> float: # Define the primary operation
    if number == 0: # Check whether division would fail
        raise ValueError("Zero is not allowed") # Raise a handled exception
    return 100 / number # Return the primary result


def fallback_operation(number: int) -> float: # Define the fallback operation
    return 0.0 # Return a safe default result


primary_runnable: RunnableLambda = RunnableLambda(primary_operation) # Wrap the primary function as a Runnable

fallback_runnable: RunnableLambda = RunnableLambda(fallback_operation) # Wrap the fallback function as a Runnable

safe_runnable: RunnableWithFallbacks[int, float] = RunnableWithFallbacks( # Create the fallback wrapper
    runnable=primary_runnable, # Set the Runnable executed first
    fallbacks=[fallback_runnable], # Set the fallback Runnables in order
    exceptions_to_handle=(ValueError,), # Trigger fallback only for ValueError
) # Finish creating the fallback wrapper

success_result: float = safe_runnable.invoke(5) # Execute the primary Runnable successfully

fallback_result: float = safe_runnable.invoke(0) # Execute the fallback after the primary fails

print(success_result) # Display the primary result

print(fallback_result) # Display the fallback result